# Notebook 4–5 (Integrado) — Alinhamento Temporal, Retornos e Saneamento Robusto

Pipeline único, na ordem correta de execução, consolidando o antigo Notebook 4
(alinhamento temporal) e o Notebook 5.2 (saneamento robusto), com o cálculo de retornos
entre eles:

**Preços alinhados → painel consolidado → matriz de retornos → winsorização robusta.**

## Decisão metodológica central: anomalias tratadas no domínio dos retornos

A versão anterior do Notebook 4 aplicava, na *Etapa VI*, uma exclusão de ativos por
**preço máximo > R\$ 1.000** (9 tickers: PDGR3, OIBR3, OIBR4, VIVR3, LUPA3, AMER3, GFSA3,
NEXP3, RCSL4). Esse critério tinha duas fragilidades: (i) operava sobre o **nível de preço**,
ao passo que toda a etapa de otimização consome **retornos**, que são invariantes a fatores
multiplicativos de ajuste; e (ii) implicava remoção discricionária de ativos — inclusive
empresas que faliram (Americanas, Oi) —, introduzindo potencial **viés de sobrevivência**
(Brown, Goetzmann & Ross, 1992).

Este notebook **transfere integralmente a detecção de anomalias para o domínio dos retornos**
(Campbell, Lo & MacKinlay, 1997). Em vez de excluir ativos por preço, aplica-se a winsorização
robusta por **Z-score modificado de Iglewicz & Hoaglin (1993)**, $|M_i|>3{,}5$, uniforme a
todos os ativos, e sinalizam-se para auditoria os retornos economicamente impossíveis
($|R|>100\%$). A exclusão por preço fica **desligada por padrão** (`EXCLUIR_PRECO_CORROMPIDO
= False`), preservada apenas como toggle auditável; suas métricas continuam sendo calculadas
e exibidas como diagnóstico, de modo que as séries de nível corrompido permanecem visíveis
mesmo sem serem descartadas.

> **Nota sobre o toggle.** Mantida a coerência return-based, recomenda-se conferir a saída do
> flag `|R|>100%` (§7): é o teste no domínio correto para decidir, com evidência, se algum dos
> 9 ativos de nível corrompido também tem retorno irrecuperável. Caso a banca prefira a exclusão
> por preço, basta `EXCLUIR_PRECO_CORROMPIDO = True`.

## 1. Configuração — caminhos e parâmetros (limiares fixados na literatura)

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --- Caminhos relativos à raiz do projeto ---
parent_path = Path.cwd().parent.parent
INPUT_DIR_MATRIZ_PRECOS = parent_path / "data" / "Matriz_Precos"
INPUT_DIR_IBOV  = parent_path / "data" / "Ibov" / "Ibov_Diario"
INPUT_DIR_CDI   = parent_path / "data" / "CDI"
INPUT_DIR_SELIC = parent_path / "data" / "Selic"
OUTPUT_DIR      = parent_path / "data" / "Tickers"
DIR_PAINEL      = parent_path / "data" /  "Painel_Dados"
DIR_RETORNOS    = parent_path / "data" /  "Retornos"
for d in (DIR_PAINEL, DIR_RETORNOS):
    d.mkdir(parents=True, exist_ok=True)

# --- Toggle da exclusão por preço (Etapa VI) ---
EXCLUIR_PRECO_CORROMPIDO = False   # default: anomalias tratadas no domínio dos retornos
LIMIAR_PRECO_MAX         = 1000.0  # usado apenas se o toggle acima for True

# --- Limiares de saneamento (padrões da literatura; não ajustar sem justificar no texto) ---
K_MAD             = 3.5     # Iglewicz & Hoaglin (1993)
C_MAD             = 0.6745  # = Phi^{-1}(0,75)
LIMITE_IMPOSSIVEL = 1.0     # |R| > 100% a.d. = retorno economicamente impossível (flag)
TRADING_DAYS      = 252

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print(f"[OK] Exclusão por preço (Etapa VI): {'LIGADA' if EXCLUIR_PRECO_CORROMPIDO else 'DESLIGADA'}")
print(f"[OK] Winsorização MAD (Iglewicz-Hoaglin): k = {K_MAD}")
print(f"[OK] Saída painel:   {DIR_PAINEL}")
print(f"[OK] Saída retornos: {DIR_RETORNOS}")

[OK] Exclusão por preço (Etapa VI): DESLIGADA
[OK] Winsorização MAD (Iglewicz-Hoaglin): k = 3.5
[OK] Saída painel:   c:\VSCodeWorkspace\TCC_Final\data\Tickers\Painel_Dados
[OK] Saída retornos: c:\VSCodeWorkspace\TCC_Final\data\Tickers\Retornos


## 2. Carregamento dos insumos (131 ativos + IBOV + CDI + SELIC)

In [12]:
def _carregar_csv(path, date_col):
    if not path.exists():
        raise FileNotFoundError(f"{path.name} não encontrado em {path.parent}.")
    return pd.read_csv(path, parse_dates=[date_col]).set_index(date_col).sort_index()

precos = _carregar_csv(INPUT_DIR_MATRIZ_PRECOS / "Matriz_precos_sanitizada.csv", "Data")
print(f"Matriz de preços: {precos.shape[1]} ativos × {precos.shape[0]} pregões "
      f"({precos.index.min().date()} → {precos.index.max().date()})")

ibov_close   = _carregar_csv(INPUT_DIR_IBOV / "ibov_diario_2010_2026.csv", "data")["ibov_close"].rename("IBOV_close")
cdi_diario   = _carregar_csv(INPUT_DIR_CDI / "cdi_diario_bcb_2010_atual.csv", "data")["cdi_diario"].rename("CDI_diario")
selic_diario = _carregar_csv(INPUT_DIR_SELIC / "selic_diario_bcb_2010_atual.csv", "data")["selic_diario"].rename("SELIC_diario")
print(f"IBOVESPA: {len(ibov_close):,} | CDI: {len(cdi_diario):,} | SELIC: {len(selic_diario):,} obs")

# Registro acumulado de exclusões (Etapas IV-V)
_excl = INPUT_DIR_MATRIZ_PRECOS / "tickers_excluidos.csv"
excluidos_prev = (pd.read_csv(_excl) if _excl.exists()
                  else pd.DataFrame(columns=["ticker","presenca_pct","primeiro_pregao_real","motivo"]))
print(f"Registro de exclusões anterior: {len(excluidos_prev)} tickers")

Matriz de preços: 131 ativos × 3967 pregões (2010-01-04 → 2025-12-30)
IBOVESPA: 3,967 | CDI: 4,019 | SELIC: 4,019 obs
Registro de exclusões anterior: 365 tickers


## 3. Etapa VI — diagnóstico de integridade de série (sem exclusão por padrão)

As métricas de integridade são sempre calculadas e exibidas. A exclusão por preço só é
**aplicada** se `EXCLUIR_PRECO_CORROMPIDO = True`; caso contrário, todos os ativos seguem para
o alinhamento e o tratamento de anomalias ocorre no domínio dos retornos (§6–7).

In [13]:
rets_diag = precos.pct_change()
integridade = pd.DataFrame({
    "preco_inicial": precos.iloc[0],
    "preco_final":   precos.iloc[-1],
    "preco_max":     precos.max(),
    "preco_min":     precos.min(),
    "amplitude":     precos.max() / precos.min(),
    "saltos_>50pct": (rets_diag.abs() > 0.50).sum(),
})

mask_corrompido = integridade["preco_max"] > LIMIAR_PRECO_MAX
print(f"=== Diagnóstico: ativos com preço máximo > R$ {LIMIAR_PRECO_MAX:,.0f} "
      f"({int(mask_corrompido.sum())}) ===")
print(integridade[mask_corrompido].sort_values("preco_max", ascending=False)
      [["preco_inicial","preco_max","preco_min","amplitude","saltos_>50pct"]].to_string())

if EXCLUIR_PRECO_CORROMPIDO:
    precos_VI = precos.drop(columns=integridade.index[mask_corrompido])
    novos = pd.DataFrame({"ticker": integridade.index[mask_corrompido],
                          "presenca_pct": np.nan, "primeiro_pregao_real": np.nan,
                          "motivo": f"preco max > R$ {LIMIAR_PRECO_MAX:.0f} (Etapa VI)"})
    excluidos_prev = pd.concat([excluidos_prev, novos], ignore_index=True)
    print(f"\n[Etapa VI LIGADA] {precos.shape[1]} → {precos_VI.shape[1]} ativos "
          f"(−{int(mask_corrompido.sum())}).")
else:
    precos_VI = precos
    print(f"\n[Etapa VI DESLIGADA] Mantidos todos os {precos_VI.shape[1]} ativos. "
          f"Anomalias serão tratadas no domínio dos retornos (§6–7).")

=== Diagnóstico: ativos com preço máximo > R$ 1,000 (9) ===
             preco_inicial             preco_max  preco_min             amplitude  saltos_>50pct
PDGR3 8,661,504,016.799999 12,097,253,755.000000   0.820000 14,752,748,481.707317             12
GFSA3        12,025.947461         12,927.532550   4.690000          2,756.403529              1
AMER3         3,863.522415         12,289.634644   3.130000          3,926.400845              3
OIBR3        11,703.943587         11,955.191035   0.050000        239,103.820700              4
VIVR3        10,547.946999         11,885.438837   0.295850         40,173.867963              5
OIBR4         6,521.765601          6,895.491686   1.580000          4,364.235245              1
LUPA3         4,777.215667          5,206.833326   0.790000          6,590.928261              7
NEXP3         1,722.570661          2,668.824770   2.890000            923.468778              1
RCSL4         1,298.728113          1,650.977483   0.165000        

## 4. Alinhamento temporal por interseção de calendário

In [14]:
calendario = (precos_VI.index
               .intersection(ibov_close.index)
               .intersection(cdi_diario.index)
               .intersection(selic_diario.index)
               .sort_values())
print(f"Calendário de interseção: {len(calendario):,} pregões "
      f"({calendario.min().date()} → {calendario.max().date()})")

gaps = pd.Series(calendario).diff().dt.days
maiores = pd.DataFrame({"data": calendario, "gap_dias": gaps.values}).nlargest(5, "gap_dias")
print("\nMaiores intervalos entre pregões consecutivos (feriados prolongados esperados):")
print(maiores.to_string(index=False))

Calendário de interseção: 3,967 pregões (2010-01-04 → 2025-12-30)

Maiores intervalos entre pregões consecutivos (feriados prolongados esperados):
      data  gap_dias
2010-02-17  5.000000
2011-03-09  5.000000
2011-04-25  5.000000
2012-02-22  5.000000
2012-12-26  5.000000


## 5. Painel consolidado e exportação

In [15]:
acoes_alinhado = precos_VI.reindex(calendario)
acoes_alinhado.columns = [f"ACAO_{c}" for c in acoes_alinhado.columns]

painel = pd.concat([
    ibov_close.reindex(calendario),
    cdi_diario.reindex(calendario),
    selic_diario.reindex(calendario),
    acoes_alinhado,
], axis=1)
painel.index.name = "data"

n_nan = int(painel.isna().sum().sum())
print(f"Painel: {painel.shape[0]:,} pregões × {painel.shape[1]} colunas "
      f"({painel.shape[1]-3} ações + IBOV + CDI + SELIC) | NaN: {n_nan}")
assert n_nan == 0, "Há NaN no painel — investigar antes de prosseguir."

painel.to_parquet(DIR_PAINEL / "painel_alinhado.parquet", engine="pyarrow")
painel.to_csv(DIR_PAINEL / "painel_alinhado.csv", date_format="%Y-%m-%d", float_format="%.8f")
print("[OK] Painel exportado (parquet + csv).")

Painel: 3,967 pregões × 134 colunas (131 ações + IBOV + CDI + SELIC) | NaN: 0
[OK] Painel exportado (parquet + csv).


## 6. Matriz de retornos (simples e logarítmica)

Retorno simples ($R_t=P_t/P_{t-1}-1$) para agregação de carteira; retorno log
($r_t=\ln(1+R_t)$) para modelagem econométrica. A primeira linha (sem pregão anterior) é
descartada. CDI/SELIC não são transformados — seguem como $r_f$, apenas realinhados.

In [16]:
cols_acoes = [c for c in painel.columns if c.startswith("ACAO_")]
precos_acoes = painel[cols_acoes]

ret_simples = precos_acoes.pct_change().iloc[1:]
ret_log     = np.log(precos_acoes).diff().iloc[1:]

ibov_ret_simples = painel["IBOV_close"].pct_change().iloc[1:]
ibov_ret_log     = np.log(painel["IBOV_close"]).diff().iloc[1:]

rf = painel[["CDI_diario","SELIC_diario"]].reindex(ret_simples.index)

print(f"Retornos: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos "
      f"({ret_simples.index.min().date()} → {ret_simples.index.max().date()})")
assert np.allclose(ret_log.values, np.log1p(ret_simples.values), atol=1e-10), "log ≠ ln(1+simples)."
print("[OK] Identidade log ≡ ln(1+simples) verificada nos retornos brutos.")

Retornos: 3966 pregões × 131 ativos (2010-01-05 → 2025-12-30)
[OK] Identidade log ≡ ln(1+simples) verificada nos retornos brutos.


## 7. Saneamento robusto — winsorização por Z-score modificado (MAD, k=3,5)

Aplicada **apenas às ações**: o benchmark e o $r_f$ ficam intactos (os dias extremos do índice
são economicamente reais — e.g., os *circuit breakers* de março/2020 — e necessários ao CAPM).
Antes, registra-se para auditoria todo ativo com $|R|>100\%$ a.d. (impossibilidade econômica).

In [17]:
# --- 7.1 Auditoria: retornos economicamente impossíveis ---
impossiveis = (ret_simples.abs() > LIMITE_IMPOSSIVEL).sum()
flag = impossiveis[impossiveis > 0].sort_values(ascending=False)
audit = pd.DataFrame({"dias_|R|>100%": flag,
                      "maior_|R|_%": (ret_simples[flag.index].abs().max()*100).round(0),
                      "preco_mediano_R$": painel[flag.index].median().round(4)})
print("=== Ativos com |R|>100% (verificar contra eventos corporativos) ===")
print(audit.to_string() if len(audit) else "Nenhum.")

# --- 7.2 Winsorização MAD ---
def cercas_mad(serie, k=K_MAD, c=C_MAD):
    x = serie.astype(float).values
    med = np.median(x); mad = np.median(np.abs(x - med))
    if mad == 0:
        return None, None
    return med - k*mad/c, med + k*mad/c

ret_log_san = ret_log.copy()
relatorio = []
for a in cols_acoes:
    lo, hi = cercas_mad(ret_log[a])
    if lo is None:
        relatorio.append({"ativo": a, "n_winsor": 0, "mad_zero": True}); continue
    n = int(((ret_log[a] < lo) | (ret_log[a] > hi)).sum())
    ret_log_san[a] = ret_log[a].clip(lower=lo, upper=hi)
    relatorio.append({"ativo": a, "n_winsor": n, "mad_zero": False})

ret_simples_san = np.expm1(ret_log_san)   # reconstrói simples (coerência garantida)
rel = pd.DataFrame(relatorio).set_index("ativo")
n_total = int(rel["n_winsor"].sum())
print(f"\n[+] Winsorizadas: {n_total} obs ({n_total/(len(cols_acoes)*len(ret_log))*100:.3f}%) "
      f"em {int((rel['n_winsor']>0).sum())} de {len(cols_acoes)} ativos "
      f"| MAD=0: {int(rel['mad_zero'].sum())}")

=== Ativos com |R|>100% (verificar contra eventos corporativos) ===
             dias_|R|>100%  maior_|R|_%  preco_mediano_R$
ACAO_FICT3               2   142.000000          0.627100
ACAO_GOLL54              2 1,847.000000          0.148000
ACAO_LUPA3               2   148.000000          3.528900
ACAO_TELB4               2   223.000000         22.509600
ACAO_AMER3               1   180.000000      1,616.798000

[+] Winsorizadas: 13673 obs (2.632%) em 131 de 131 ativos | MAD=0: 0


## 8. Testes de integridade finais

In [18]:
print("[+] Verificando integridade...\n")
assert ret_log_san.index.equals(ret_log.index), "Falha: índice alterado."
assert ret_log_san.shape == ret_simples_san.shape == ret_log.shape, "Falha: shape."
print("      [OK] Índice e dimensões preservados")
assert ret_log_san.isna().sum().sum() == 0 and ret_simples_san.isna().sum().sum() == 0, "Falha: NaN."
assert np.isfinite(ret_log_san.values).all() and np.isfinite(ret_simples_san.values).all(), "Falha: inf."
print("      [OK] Sem NaN ou infinitos")
assert np.allclose(ret_log_san.values, np.log1p(ret_simples_san.values), atol=1e-10), "Falha: identidade."
print("      [OK] Identidade log ≡ ln(1+simples) preservada")
mask_validos = ~rel["mad_zero"].values
resid = int((ret_simples_san.loc[:, mask_validos].abs() > LIMITE_IMPOSSIVEL).sum().sum())
print(f"      [OK] |R|>100% remanescentes (ativos com MAD>0): {resid}")
print("\n[OK] Matrizes saneadas aprovadas.")

[+] Verificando integridade...

      [OK] Índice e dimensões preservados
      [OK] Sem NaN ou infinitos
      [OK] Identidade log ≡ ln(1+simples) preservada
      [OK] |R|>100% remanescentes (ativos com MAD>0): 0

[OK] Matrizes saneadas aprovadas.


## 9. Exportação dos artefatos finais

In [19]:
def _salvar(df, nome, ddir=DIR_RETORNOS):
    df.to_csv(ddir / f"{nome}.csv", index=True, date_format="%Y-%m-%d", float_format="%.8f")
    try:
        df.to_parquet(ddir / f"{nome}.parquet", engine="pyarrow")
        print(f"  {nome}.csv + .parquet")
    except Exception as e:
        print(f"  {nome}.csv  [parquet: {e}]")

print("[+] Gravando em:", DIR_RETORNOS, "\n")
_salvar(ret_simples,     "retornos_simples")            # brutos (referência)
_salvar(ret_log,         "retornos_log")
_salvar(ret_simples_san, "retornos_simples_saneado")    # saneados (insumo do NB6+)
_salvar(ret_log_san,     "retornos_log_saneado")
_salvar(pd.DataFrame({"ibov_ret_simples": ibov_ret_simples,
                      "ibov_ret_log": ibov_ret_log}), "ibov_retornos")
_salvar(rf.rename(columns=str.lower), "rf_diario")

rel.to_csv(DIR_RETORNOS / "log_winsorizacao.csv", float_format="%.8f")
if len(audit): audit.to_csv(DIR_RETORNOS / "flag_integridade_eventos.csv", float_format="%.4f")
pd.Series([c.replace("ACAO_","") for c in cols_acoes], name="ticker").to_csv(
    OUTPUT_DIR / "tickers_finais.csv", index=False)

# Registro de exclusões: só é alterado se a Etapa VI estiver ligada
excluidos_prev.to_csv(OUTPUT_DIR / "tickers_excluidos.csv", index=False)

print("\n" + "="*62)
print("PIPELINE INTEGRADO CONCLUÍDO")
print("="*62)
print(f"Ativos no painel: {len(cols_acoes)}  |  Pregões de retorno: {ret_simples_san.shape[0]}")
print(f"Exclusão por preço: {'LIGADA' if EXCLUIR_PRECO_CORROMPIDO else 'DESLIGADA (anomalias no domínio dos retornos)'}")
print(f"Observações winsorizadas: {n_total} ({n_total/(len(cols_acoes)*len(ret_log))*100:.3f}%)")
print("Insumo pronto: retornos_*_saneado → momentos/covariância e otimização (NB 6+).")
print("="*62)

[+] Gravando em: c:\VSCodeWorkspace\TCC_Final\data\Tickers\Retornos 

  retornos_simples.csv + .parquet
  retornos_log.csv + .parquet
  retornos_simples_saneado.csv + .parquet
  retornos_log_saneado.csv + .parquet
  ibov_retornos.csv + .parquet
  rf_diario.csv + .parquet

PIPELINE INTEGRADO CONCLUÍDO
Ativos no painel: 131  |  Pregões de retorno: 3966
Exclusão por preço: DESLIGADA (anomalias no domínio dos retornos)
Observações winsorizadas: 13673 (2.632%)
Insumo pronto: retornos_*_saneado → momentos/covariância e otimização (NB 6+).


## 10. Apêndice para o TCC — texto de redação

> *"As séries de preços, do índice de mercado e da taxa livre de risco foram alinhadas pela
> interseção de seus calendários de negociação, produzindo um painel balanceado sem dados
> ausentes. O tratamento de anomalias foi conduzido integralmente no domínio dos retornos —
> e não por exclusão de ativos com base no nível de preço, evitando tanto a discricionariedade
> na seleção de tickers quanto o viés de sobrevivência (Brown, Goetzmann e Ross, 1992).
> Sobre os log-retornos de cada ação aplicou-se o Z-score modificado de Iglewicz e Hoaglin
> (1993), $M_i = 0{,}6745\,(r_i-\tilde r)/\mathrm{MAD}$, winsorizando-se as observações com
> $|M_i|>3{,}5$; o índice e a taxa livre de risco, cujos extremos são economicamente
> legítimos, foram preservados. Ativos com retorno diário economicamente impossível
> ($|R|>100\%$) foram sinalizados para verificação contra o calendário de eventos
> corporativos. Todo o procedimento foi registrado em logs auditáveis."*

**Artefatos em `data/`:** `Painel_Dados/painel_alinhado.*`; `Retornos/retornos_{simples,log}.*`
(brutos), `Retornos/retornos_{simples,log}_saneado.*` (insumo dos otimizadores),
`Retornos/ibov_retornos.*`, `Retornos/rf_diario.*`, `Retornos/log_winsorizacao.csv`,
`Retornos/flag_integridade_eventos.csv` e `tickers_finais.csv`.